# KMeans 
310_kmeans_clustering.ipynb

K = 13 clusters (fixed)

Raw ERSP matrices (300×128) from
\01_FBM_Analysis\outputs\04_ersp_LM_RAWONLY\PAT_3301\LM\ERSP_matrix\picture\

Your three example files like:
PAT_3301_picture_None_ERSP_AG1_TN.npy, AG2, AG3, etc.

In [ ]:
## 0. Config & imports
import os
from pathlib import Path
import datetime
import json

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

import matplotlib.pyplot as plt  # for optional plots (no seaborn)

# ---- Identity / tags ----
SCRIPT_NAME = "310_kmeans_clustering.ipynb"
ALGO_TAG = "kmeans_raw"
RUN_ID = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")  # e.g. 20251117_112233

# ---- Paths ----
ANALYSIS_ROOT = Path(r"\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda")

# Input: RAWONLY ERSP matrices
INPUT_DIR = ANALYSIS_ROOT / r"01_FBM_Analysis/outputs/04_ersp_LM_RAWONLY"

# Output: clustering results & figures
CLUSTERING_OUTPUT_DIR  = ANALYSIS_ROOT / r"02_FBM_Clustering/outputs/clustering/kmeans"
FIGURES_EMBEDDINGS_DIR = ANALYSIS_ROOT / r"02_FBM_Clustering/outputs/figures/embeddings"
FIGURES_ERSP_DIR       = ANALYSIS_ROOT / r"02_FBM_Clustering/outputs/figures/ersp_clusters"
LOG_DIR                = ANALYSIS_ROOT / r"02_FBM_Clustering/outputs/logs"

for d in [CLUSTERING_OUTPUT_DIR, FIGURES_EMBEDDINGS_DIR, FIGURES_ERSP_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)


# WM electrodes for double check    
# ---- WM reference maps (unchanged data, compact helpers) ----
WHITE_MATTER_REFS_NUMS = {
    "PAT_3066": {"FIG":[6,5], "FOG":[15], "FOD":[12], "CAG":[11,10,9,8,7], "CAD":[8,6,5,3,2],
                 "IAG":[18,11,9,3,2], "IMG":[18,9,8,7,6,3], "AG":[3,2,1], "HAG":[1], "HPG":[7,2],
                 "PPG":[6,5,4,2], "TIG":[3], "AD":[8,6,4,3,2,1], "HAD":[3,2,1], "HPD":[2,1], "PPD":[6,5,4,3]},
    "PAT_3975": {"FOG":[15,10,8,3,1], "CPG":[10,6,5,4,3], "AG":[12], "HAG":[5,2,1], "PHG":[6,4],
                 "IAG":[8,6,4,3,2,1], "IMG":[13,12,11,6,3], "FOD":[14,13,12,11,9,6,5,3],
                 "CPD":[15,8,6,2], "AD":[7,4,3,2,1], "HAD":[3,2,1], "PHD":[11,7], "IAD":[10],
                 "IMD":[17,16,15,14,12,10,9,7]},
    "PAT_3415": {"IMG":[16,15,12,11,8,6,5,4,3,2], "IPG":[12,10,8,3,2,1], "HLG":[17,16,3,2]},
    "PAT_3965": {"ag":[3,4,5,6], "hpg":[8,9,10,11], "imd":[6,7,8,9,10,11,12]},
    "PAT_2868": {"POP":[3], "IDM":[4,1], "SMA":[2,1], "PPS":[3,2,1], "PRI":[3,2,1], "POM":[4,1],
                 "PPI":[2], "POI":[3,2,1]},
    "PAT_3455": {"AD":[2,1], "HAD":[2,1], "TOD":[6], "CPD":[12,11,10,9,6,5,2], "OPD":[5,4,3,2],
                 "PHD":[5], "OTD":[8], "TSP":[8,6,5,4], "IMD":[11,5,4,3], "IPD":[14,11]},
    "PAT_3390": {"IAG":[18,17,16,9,1], "CAG":[8,6,5,4,2], "CPG":[15,13,10,9,1], "HPG":[10,9,8],
                 "AG":[12,11], "HAG":[12,11,10,9], "FOG":[9,4,3,2,1]},
    "MicroEPI-B-01": {"pI_L":[8,9,10,11,12,13,14], "A_L":[4,5],"ITG_L":[6,9],"sSMG_L":[2,3,5,6],"IOG_L":[4,5,7],"LinG_L":[4,5]},
    "EL042": {"OF_R":[10,11],"aI_R":[1,2,3,4,5,6,7,10,11,12,13,14,17],"A_R":[5,6,7],"aH_R":[7],"pH_R":[7,8],"STG_R":[3],
             "A_L":[6,7,8,9]},
    "EL040":{"FP_R":[8],"OF_R":[4,5,9,10,11,12,13,14,15],"CinG_R":[5,6,7,8,9,10,11,12],"aI_R":[9,10,11,12,13,14,15],
             "A_R":[8,9],"EntG_R":[5,6,8],"aH_R":[8],"PHG_R":[5,6,7],"pH_R":[4,5,6,7,8,9],"STG_R":[1,8],"iNH_R":[13,14,15],
             "CinG_L":[5,6,7,8,9,10],"pPVH_L":[7,8,9]},
    "EL039":{"OF_R":[10,11,12,13],"pSTG_R":[2],"CinG_R":[2,3,4,5,6],"pH_L":[6,7],"LinG_R":[9],
             "STO_R":[6,7,8,9],"aSTG_R":[2],"A_L":[4,5,6]},
    "EL038":{"OF_R":[10,11,12],"CinG_R":[2,3,4,5,6],"aSTG_R":[2],"pSTG_R":[2],"STO_R":[6,7,8,9],"A_L":[4,5,6],"pH_L":[6,7]},
    "EL037":{"A_L":[5],"EntG_L":[7],"pH_L":[6],"CinG_L":[3,4,5,6,7,8,9,10,11],"aI_L":[12,13,14,15],"pI_L":[11,12],"A_R":[7,8]},
    "EL036":{"A_R":[4,5,6,7],"aH_R":[5,6],"mH_R":[10],"PfC_R":[5]},
    "EL035":{"A_L":[8],"aH_L":[7],"A_R":[6,7,8],"aH_R":[7,8],"pH_R":[4,5,6,7],"ParaHG_R":[7,8,9],"OF_R":[4,5,6,7,8,9,10],"CinG_R":[4,5,6],"aI_R":[8,9,10,11],"pI_R":[11,12,13]},
    "EL034":{"CinG_R":[4,5,6,8,9],"OFG_L":[4,5],"pSFG_L":[4],"aH_L":[6,7],"MFG_L":[3]},
    "EL030":{"A-L":[6,7,8],"aH-L":[6,7],"A-R":[7,8],"aH-R":[6,7],"pH-R":[3,4,5]},
}


# ---- Data shape constants ----
N_TIME = 300
N_FREQ = 129
N_FEATURES = N_TIME * N_FREQ

# ---- Clustering parameters ----
N_CLUSTERS = 13
RANDOM_STATE = 42

print("SCRIPT_NAME:", SCRIPT_NAME)
print("ALGO_TAG   :", ALGO_TAG)
print("RUN_ID     :", RUN_ID)
print("Input dir  :", INPUT_DIR)


## Auto-detect patient IDs from outputs/04_ersp_LM_RAWONLY

from pathlib import Path

# Patient folders = all directories inside INPUT_DIR
PATIENT_IDS = sorted([d.name for d in INPUT_DIR.iterdir() if d.is_dir()])

print("Detected patient folders:")
for p in PATIENT_IDS:
    print("  -", p)

print("\nTotal patients detected:", len(PATIENT_IDS))


In [ ]:
## 2. Helper: parse electrode name from filename

def parse_electrode_from_filename(fname: str) -> str:
    """
    Example filename:
        PAT_3301_picture_None_ERSP_AG2_TN.npy
    We want to extract 'AG2' as electrode name.
    """
    name = Path(fname).name
    # Split on '_ERSP_' first
    if "_ERSP_" in name:
        left, right = name.split("_ERSP_", 1)
        # right is like "AG2_TN.npy"
        # remove suffix and trailing parts after electrode
        right_no_ext = right.rsplit(".", 1)[0]  # remove .npy
        # Typically pattern is "<electrode>_TN" or similar
        electrode = right_no_ext.split("_")[0]
        return electrode
    else:
        # fallback: no ERSP marker, just strip extension
        return name.rsplit(".", 1)[0]


In [ ]:
## 3. Load ERSP matrices and metadata from ALL discovered patients AND ALL their conditions

TASK = "LM"   # keep this as before

ersp_list = []
meta_rows = []

for pat in PATIENT_IDS:
    patient_dir = INPUT_DIR / pat / TASK / "ERSP_matrix"
    if not patient_dir.exists():
        print(f"[WARN] Missing ERSP_matrix folder for patient {pat}: {patient_dir}")
        continue

    # Find all condition folders inside ERSP_matrix
    condition_dirs = [d for d in patient_dir.iterdir() if d.is_dir()]

    if not condition_dirs:
        print(f"[WARN] No condition subfolders for {pat} in {patient_dir}")
        continue

    print(f"\nPatient {pat} — conditions found:", [d.name for d in condition_dirs])

    for cond_dir in condition_dirs:
        cond_name = cond_dir.name
        npy_files = sorted(cond_dir.glob("*.npy"))

        if not npy_files:
            print(f"[WARN] No .npy files found in {cond_dir}")
            continue

        print(f"  Loading {len(npy_files)} files from condition '{cond_name}'")

        for fpath in npy_files:
            arr = np.load(fpath)

            if arr.shape != (N_FREQ,N_TIME):
                print(f"  [WARN] Incorrect shape {arr.shape} in {fpath}, skipping.")
                continue

            ersp_list.append(arr)

            meta_rows.append({
                "patient_id": pat,
                "condition": cond_name,
                "task": TASK,
                "electrode": parse_electrode_from_filename(fpath.name),
                "file_path": str(fpath),
            })

df_meta = pd.DataFrame(meta_rows)
df_meta.index.name = "sample_idx"
df_meta.reset_index(inplace=True)

print("\n=== Finished Loading ===")
print("Total ERSP samples loaded:", len(ersp_list))

In [ ]:
## 4. Exclude non-neural channels (PHOTO, X*, E*, ECG, AUDIO, markers, +/-) and WM refs

import re
import numpy as np
import pandas as pd

# ---- 4.1 Helper: non-neural / aux channel name filter ----

def is_non_neural_electrode(label: str) -> bool:
    """
    Returns True if the electrode name looks like:
    - PHOTO / photo
    - MRK / MKR
    - ECG
    - AUDIO
    - X1..X8, E1..E8, X
    - contains '+' or '-'
    Case-insensitive.
    """
    if label is None:
        return False
    s = str(label).strip()
    s_up = s.upper()

    # Any + or - in the name -> drop
    if "+" in s_up or "-" in s_up:
        return True

    # Direct tokens
    direct_tokens = ["PHOTO", "MRK", "MKR", "ECG", "AUDIO"]
    if any(tok in s_up for tok in direct_tokens):
        return True

    # Exactly X
    if s_up == "X":
        return True

    # Exactly X1..X8 or E1..E8
    if re.fullmatch(r"[XE][1-8]", s_up):
        return True

    return False

# ---- 4.2 Build WM-ref electrode sets per patient ----
# WHITE_MATTER_REFS_NUMS must already be defined in your config cell.

WM_REF_SET = {}
for pat, grids in WHITE_MATTER_REFS_NUMS.items():
    names = set()
    for prefix, nums in grids.items():
        for num in nums:
            names.add(f"{prefix}{num}".upper())
    WM_REF_SET[pat] = names

# ---- 4.3 Masks: non-neural + WM-ref ----

# Non-neural mask based on name
non_neural_mask = df_meta["electrode"].apply(is_non_neural_electrode)

# WM-ref mask based on patient + electrode name
def is_wm_ref(row):
    pat = row["patient_id"]
    label = str(row["electrode"]).upper()
    if pat not in WM_REF_SET:
        return False
    return label in WM_REF_SET[pat]

wm_ref_mask = df_meta.apply(is_wm_ref, axis=1)

exclude_mask = non_neural_mask | wm_ref_mask
keep_mask = ~exclude_mask

print(f"Total samples before exclusion: {len(df_meta)}")
print(f"  Non-neural channels to exclude: {non_neural_mask.sum()}")
print(f"  WM-ref channels to exclude:     {wm_ref_mask.sum()}")
print(f"  Total excluded:                  {exclude_mask.sum()}")
print(f"  Remaining for clustering:        {keep_mask.sum()}")

# ---- 4.4 Show what is being excluded ----
excluded_df = df_meta[exclude_mask].copy()
print("\nExamples of excluded electrodes:")
display(
    excluded_df[["patient_id", "electrode", "condition"]]
    .sort_values(["patient_id", "electrode"])
    .head(50)
)

# ---- 4.5 Apply exclusion to df_meta and ersp_list ----
# sample_idx is the original index used when building ersp_list

bad_sample_idxs = df_meta.loc[exclude_mask, "sample_idx"].to_numpy()
keep_sample_idxs = df_meta.loc[keep_mask, "sample_idx"].to_numpy()

# Rebuild ersp_list in the new order (only kept indices)
ersp_list = [ersp_list[i] for i in keep_sample_idxs]

# Filter df_meta and reset index
df_meta = df_meta[keep_mask].reset_index(drop=True)

print("\nAfter exclusion:")
print("  df_meta rows:", len(df_meta))
print("  ersp_list len:", len(ersp_list))


In [ ]:
## 4.x High-activity electrode screening & visualization (pos + neg thresholds)

import numpy as np
import matplotlib.pyplot as plt

# --- Parameters ---
thr_pos = 2.2        # positive threshold
min_prop_pos = 0.02  # min proportion (> thr_pos) to be considered high-positive

thr_neg = -3       # negative threshold
min_prop_neg = 0.04  # min proportion (< thr_neg) to be considered high-negative

max_examples = 20    # how many electrodes to plot from each group
n_bins = 1000        # histogram bins for value distributions

if len(ersp_list) == 0:
    raise RuntimeError("No ERSP data available after exclusions.")

# --- 1) Compute proportions above/below thresholds per electrode ---
prop_pos = []
prop_neg = []
high_flags = []

for arr in ersp_list:
    vals = arr  # shape (N_FREQ, N_TIME)

    mask_pos = vals > thr_pos
    mask_neg = vals < thr_neg

    p_pos = mask_pos.mean()
    p_neg = mask_neg.mean()

    prop_pos.append(p_pos)
    prop_neg.append(p_neg)

    high_flags.append((p_pos >= min_prop_pos) or (p_neg >= min_prop_neg))

df_meta["prop_above_pos"] = prop_pos
df_meta["prop_below_neg"] = prop_neg
df_meta["high_activity"] = high_flags

n_high = df_meta["high_activity"].sum()
n_low  = len(df_meta) - n_high

print(f"Positive threshold: value > {thr_pos}, high if prop ≥ {min_prop_pos:.2f}")
print(f"Negative threshold: value < {thr_neg}, high if prop ≥ {min_prop_neg:.2f}")
print(f"High-activity electrodes: {n_high}")
print(f"Normal/low electrodes:    {n_low}")

# --- 2) Visualize a few ERSPs from each group (high vs low) ---

high_idx = df_meta.index[df_meta["high_activity"]].tolist()
low_idx  = df_meta.index[~df_meta["high_activity"]].tolist()

print("\nExample high-activity electrodes:")
display(
    df_meta.loc[high_idx, ["patient_id", "electrode", "condition",
                           "prop_above_pos", "prop_below_neg"]]
    .sort_values(["prop_above_pos", "prop_below_neg"], ascending=[False, False])
    .head(min(max_examples, len(high_idx)))
)

print("\nExample normal/low electrodes:")
display(
    df_meta.loc[low_idx, ["patient_id", "electrode", "condition",
                          "prop_above_pos", "prop_below_neg"]]
    .sort_values(["prop_above_pos", "prop_below_neg"], ascending=[True, True])
    .head(min(max_examples, len(low_idx)))
)

# Determine common color scale from subset for visualization
all_vals_sample = []
for i in (high_idx[:max_examples] + low_idx[:max_examples]):
    all_vals_sample.append(ersp_list[i].ravel())
if all_vals_sample:
    all_vals_sample = np.concatenate(all_vals_sample)
    v = np.nanmax(np.abs(all_vals_sample))
    vmin, vmax = -v, v
else:
    vmin = vmax = None

def plot_group_examples(indices, title_prefix):
    n = min(max_examples, len(indices))
    if n == 0:
        print(f"No electrodes in group '{title_prefix}', skipping plots.")
        return

    n_cols = 3
    n_rows = int(np.ceil(n / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3 * n_rows))
    axes = np.array(axes).reshape(-1)

    for ax, idx in zip(axes, indices[:n]):
        ersp = ersp_list[idx]
        meta = df_meta.iloc[idx]

        im = ax.imshow(
            ersp,
            aspect="auto",
            origin="lower",
            cmap="bwr",
            vmin=-5,
            vmax=5,
            interpolation="nearest",
        )
        ax.set_title(
            f"{meta['patient_id']} {meta['electrode']} {meta['condition']}\n"
            f"prop>+{thr_pos}={meta['prop_above_pos']:.2f}, "
            f"prop<{thr_neg}={meta['prop_below_neg']:.2f}",
            fontsize=8,
        )
        ax.set_xlabel("Time (samples)", fontsize=7)
        ax.set_ylabel("Freq", fontsize=7)

    for ax in axes[n:]:
        ax.axis("off")

    fig.suptitle(f"{title_prefix} electrodes", fontsize=12)
    plt.tight_layout()
    plt.show()

# Plot examples
plot_group_examples(high_idx, "High-activity")
plot_group_examples(low_idx,  "Normal/low")

# --- 3) Compare value distributions (high vs low) without giant arrays ---

glob_min, glob_max = np.inf, -np.inf
for arr in ersp_list:
    m = np.nanmin(arr)
    M = np.nanmax(arr)
    if m < glob_min:
        glob_min = m
    if M > glob_max:
        glob_max = M

bins = np.linspace(glob_min, glob_max, n_bins + 1)
high_hist = np.zeros(n_bins, dtype=np.float64)
low_hist  = np.zeros(n_bins, dtype=np.float64)

for idx, arr in enumerate(ersp_list):
    vals = arr.ravel()
    h, _ = np.histogram(vals, bins=bins)
    if df_meta.iloc[idx]["high_activity"]:
        high_hist += h
    else:
        low_hist += h

high_hist = high_hist / high_hist.sum() if high_hist.sum() > 0 else high_hist
low_hist  = low_hist  / low_hist.sum()  if low_hist.sum()  > 0 else low_hist

bin_centers = 0.5 * (bins[:-1] + bins[1:])

plt.figure(figsize=(8, 5))
plt.plot(bin_centers, high_hist, label="High-activity electrodes", linewidth=2)
plt.plot(bin_centers, low_hist,  label="Normal/low electrodes", linewidth=2, linestyle="--")
plt.xlabel("ERSP value")
plt.ylabel("Probability")
plt.xlim((-10, 10))
plt.title(
    "Value distribution: high vs low electrodes\n"
    f"(> {thr_pos} with prop≥{min_prop_pos:.2f} OR "
    f"< {thr_neg} with prop≥{min_prop_neg:.2f})"
)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
## 4.y Filter out low-activity electrodes (keep only high-activity)

if "high_activity" not in df_meta.columns:
    raise RuntimeError("Column 'high_activity' not found. Run the threshold cell first.")

# Determine indices to keep
keep_idx = df_meta.index[df_meta["high_activity"]].tolist()
drop_idx = df_meta.index[~df_meta["high_activity"]].tolist()

print("====== High-Activity Electrode Filtering ======")
print(f"Total electrodes before filtering: {len(df_meta)}")
print(f"High-activity electrodes kept:    {len(keep_idx)}")
print(f"Low-activity electrodes removed:   {len(drop_idx)}")
print("===============================================")



# Apply filtering to df_meta and ersp_list
df_meta = df_meta.loc[keep_idx].reset_index(drop=True)
ersp_list = [ersp_list[i] for i in keep_idx]

print("\nAfter filtering:")
print("  df_meta rows:", len(df_meta))
print("  ersp_list len:", len(ersp_list))

# Optional: rebuild sample_idx if you use it later
df_meta["sample_idx"] = np.arange(len(df_meta))


In [ ]:
## 3.2. UMAP embedding: inspect all samples 
#                      to decide which ones to remove

import numpy as np
import matplotlib.pyplot as plt
import umap
from sklearn.utils import validation as skl_validation

# --- Safety checks ---
if len(ersp_list) == 0:
    raise RuntimeError("No ERSP data loaded, cannot run UMAP.")

# Make sure UMAP uses the modern sklearn check_array (fix for ensure_all_finite issue)
umap.umap_.check_array = skl_validation.check_array

# --- In-place NaN fix on ERSPs (freq x time) ---
for arr in ersp_list:
    np.nan_to_num(arr, nan=0.0, copy=False)

# --- Build feature matrix directly: (n_samples, N_FREQ * N_TIME) as float32 ---
n_samples = len(ersp_list)
X_raw = np.empty((n_samples, N_FREQ * N_TIME), dtype=np.float32)

for i, arr in enumerate(ersp_list):
    # arr shape: (N_FREQ, N_TIME) = (129, 300)
    X_raw[i, :] = arr.reshape(-1)

print("X_raw shape for UMAP:", X_raw.shape)

# --- Run UMAP on raw (flattened) data ---
umap_model = umap.UMAP(
    n_neighbors=15,
    min_dist=0.1,
    n_components=2,
    metric="euclidean",
    random_state=42,
)

embedding = umap_model.fit_transform(X_raw)
print("UMAP embedding shape:", embedding.shape)

# Attach embedding directly to df_meta (no extra copy)
df_meta["UMAP_1"] = embedding[:, 0]
df_meta["UMAP_2"] = embedding[:, 1]

# --------------------------------------------------------------------
# Plot 1: UMAP colored by patient, with electrode labels
# --------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(10*2, 10*2))  # ~4x bigger area than 6x5

patient_codes, patient_uniques = pd.factorize(df_meta["patient_id"])
sc = ax.scatter(
    df_meta["UMAP_1"],
    df_meta["UMAP_2"],
    c=patient_codes,
    s=15,
    alpha=0.7,
)

ax.set_title("UMAP – colored by patient_id", fontsize=14)
ax.set_xlabel("UMAP 1", fontsize=12)
ax.set_ylabel("UMAP 2", fontsize=12)

# Add electrode name as text next to each point
for x, y, label in zip(df_meta["UMAP_1"], df_meta["UMAP_2"], df_meta["electrode"]):
    ax.text(x, y, str(label), fontsize=5, alpha=0.7)

# Legend for patients
handles = []
labels = []
cmap = sc.cmap
norm = plt.Normalize(vmin=patient_codes.min(), vmax=patient_codes.max())

for code, pid in enumerate(patient_uniques):
    handles.append(
        plt.Line2D(
            [], [], marker="o", linestyle="", markersize=6,
            color=cmap(norm(code))
        )
    )
    labels.append(pid)

ax.legend(handles, labels, title="patient_id", fontsize=8, loc="best")
plt.tight_layout()
plt.show()



In [ ]:
## 4. Build X_raw (flattened) and standardize

if len(ersp_list) == 0:
    raise RuntimeError("No ERSP data loaded. Check paths and file patterns.")

# Stack into (n_samples, N_TIME, N_FREQ)
X_3d = np.stack(ersp_list, axis=0)  # shape: (n_samples, 300, 128)
print("X_3d shape:", X_3d.shape)

# Flatten each matrix into a single vector (row-major)
X_raw = X_3d.reshape(X_3d.shape[0], -1)  # shape: (n_samples, 38400)
print("X_raw shape:", X_raw.shape)

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)
print("X_scaled shape:", X_scaled.shape)

# Quick sanity check on a subset of features
print("Mean (first 10 features):", X_scaled[:,:10].mean(axis=0))
print("Std  (first 10 features):", X_scaled[:, :10].std(axis=0))


# 1. KMEANS

In [ ]:
# ## 5. Fit K-means with fixed K = 13
# N_CLUSTERS_LOOP = [3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,19,20,21,22,23,24,25]
# sil_score=[]

# for N_CLUSTERS in N_CLUSTERS_LOOP:
#     kmeans = KMeans(
#         n_clusters=N_CLUSTERS,
#         init="k-means++",
#         n_init=20,
#         max_iter=300,
#         random_state=RANDOM_STATE,
#         verbose=0,
#     )

#     kmeans.fit(X_scaled)
#     labels = kmeans.labels_  # shape: (n_samples,)

#     print("K-means finished.")
#     print("Inertia        :", kmeans.inertia_)
#     print("Cluster counts :", np.bincount(labels))

#     # If at least 2 clusters are non-empty, compute silhouette
#     if len(np.unique(labels)) > 1:
#         sil = silhouette_score(X_scaled, labels)
#     else:
#         sil = np.nan
#     print("Silhouette score:", sil)
#     sil_score.append(sil)
    
    
    
#     ## 5.1 Visualize K means: 
#     centers = kmeans.cluster_centers_        # shape: (N_CLUSTERS, N_FREQ*N_TIME)

#     # Compute symmetric color limits around 0 for nicer bwr visualization
#     c_min = centers.min()
#     c_max = centers.max()
#     c_abs = max(abs(c_min), abs(c_max))
#     vmin, vmax = -c_abs, c_abs

#     n_clusters = centers.shape[0]

#     # Layout: 3 rows x 5 cols is enough for 13 clusters
#     n_cols = 5
#     n_rows = int(np.ceil(n_clusters / n_cols))

#     fig, axes = plt.subplots(n_rows, n_cols, figsize=(3*n_cols, 2.5*n_rows))
#     axes = np.array(axes).reshape(-1)  # flatten

#     for c in range(n_clusters):
#         ax = axes[c]
#         centroid_flat = centers[c]                     # (N_FREQ*N_TIME,)
#         centroid_ersp = centroid_flat.reshape(N_FREQ, N_TIME)  # (129, 300)

#         im = ax.imshow(
#             centroid_ersp,
#             aspect="auto",
#             origin="lower",
#             cmap="bwr",
#             vmin=-5,
#             vmax=5,
#             interpolation="nearest",
#         )

#         n_c = np.sum(labels == c)
#         ax.set_title(f"Cluster {c} (n={n_c})", fontsize=8)
#         ax.set_xlabel("Time (samples)", fontsize=7)
#         ax.set_ylabel("Freq", fontsize=7)

#     # Hide any unused axes
#     for ax in axes[n_clusters:]:
#         ax.axis("off")

#     # Single colorbar for all centroids
#     #cbar = fig.colorbar(im, ax=axes.tolist(), shrink=0.7, pad=0.02)
#     #cbar.set_label("Centroid value (scaled feature space)", fontsize=8)

#     plt.tight_layout()
#     plt.show()
#     plt.close()


## 5. Fit K-means loop
from sklearn.metrics import silhouette_score, silhouette_samples
import json, os

N_CLUSTERS_LOOP = [3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,19,20,21,22,23,24,25,27,29,31,33]
sil_score = []

for N_CLUSTERS in N_CLUSTERS_LOOP:
    kmeans = KMeans(
        n_clusters=N_CLUSTERS,
        init="k-means++",
        n_init=20,
        max_iter=300,
        random_state=RANDOM_STATE,
        verbose=0,
    )
    kmeans.fit(X_scaled)
    labels = kmeans.labels_  # shape: (n_samples,)
    print("K-means finished.")
    print("Inertia        :", kmeans.inertia_)
    print("Cluster counts :", np.bincount(labels))

    # ── Overall silhouette ──
    if len(np.unique(labels)) > 1:
        sil = silhouette_score(X_scaled, labels)
        # Per-cluster silhouette (mean of sample silhouettes within each cluster)
        sil_samples = silhouette_samples(X_scaled, labels)
        sil_per_cluster = {
            int(i): float(sil_samples[labels == i].mean())
            for i in range(N_CLUSTERS)
        }
    else:
        sil = np.nan
        sil_per_cluster = {}

    print("Silhouette score (avg)      :", sil)
    print("Silhouette score (per cluster):")
    for k, v in sil_per_cluster.items():
        print(f"  Cluster {k:>2}: {v:+.4f}")
    sil_score.append(sil)

    # ── Save metrics JSON ──
    run_id = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
    algo_tag = f"kmeans_raw"
    metrics = {
        "script_name": "310_kmeans_clustering.ipynb",
        "algo_tag": algo_tag,
        "run_id": run_id,
        "n_clusters": int(N_CLUSTERS),
        "n_samples": int(X_scaled.shape[0]),
        "n_features": int(X_scaled.shape[1]),
        "inertia": float(kmeans.inertia_),
        "silhouette": float(sil) if not np.isnan(sil) else None,
        "silhouette_per_cluster": sil_per_cluster,   # ← new
        "cluster_counts": [int(x) for x in np.bincount(labels)],
        "input_root": str(INPUT_DIR),
        "patients": PATIENT_IDS,
        "task": TASK,
        "random_state": int(RANDOM_STATE),
        "note": f"Clustering on raw flattened ERSP (300x128) without PCA. K={N_CLUSTERS}."
    }
    out_dir = CLUSTERING_OUTPUT_DIR
    out_dir.mkdir(parents=True, exist_ok=True)
    metrics_path = out_dir / f"metrics_{algo_tag}_{N_CLUSTERS}k_{run_id}.json"
    with open(metrics_path, "w") as f:
        json.dump(metrics, f, indent=2)
    print(f"Metrics saved → {metrics_path}")

    ## 5.1 Visualize K-means centroids
    centers = kmeans.cluster_centers_
    c_abs = max(abs(centers.min()), abs(centers.max()))
    n_cols = 5
    n_rows = int(np.ceil(N_CLUSTERS / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3*n_cols, 2.5*n_rows))
    axes = np.array(axes).reshape(-1)
    for c in range(N_CLUSTERS):
        ax = axes[c]
        centroid_ersp = centers[c].reshape(N_FREQ, N_TIME)
        ax.imshow(
            centroid_ersp,
            aspect="auto", origin="lower", cmap="bwr",
            vmin=-5, vmax=5, interpolation="nearest",
        )
        sil_c = sil_per_cluster.get(c, float("nan"))
        n_c = np.sum(labels == c)
        ax.set_title(f"C{c} n={n_c} sil={sil_c:+.3f}", fontsize=7)
        ax.set_xlabel("Time", fontsize=6)
        ax.set_ylabel("Freq", fontsize=6)
    for ax in axes[N_CLUSTERS:]:
        ax.axis("off")
    plt.suptitle(f"K={N_CLUSTERS}  |  avg silhouette={sil:.4f}", fontsize=9)
    plt.tight_layout()
    plt.show()
    plt.close()

In [ ]:
plt.figure(figsize=(5,3))

# Line + scatter (black & white)
plt.plot(
    N_CLUSTERS_LOOP,
    sil_score,
    marker="o",
    linestyle="-",
    color="black",
    markersize=5
)

plt.xlabel("Number of clusters (K)")
plt.ylabel("Silhouette score")
plt.title("Silhouette score vs K")
plt.tight_layout()
plt.show()


In [ ]:
## 6. Attach labels to metadata and save outputs

df_meta["cluster_kmeans_raw"] = labels

# Paths for this run
labels_path = CLUSTERING_OUTPUT_DIR / f"labels_{ALGO_TAG}_{RUN_ID}.csv"
metrics_path = CLUSTERING_OUTPUT_DIR / f"metrics_{ALGO_TAG}_{RUN_ID}.json"

print("Saving labels to:", labels_path)
df_meta.to_csv(labels_path, index=False)

# Build metrics dict
metrics = {
    "script_name": SCRIPT_NAME,
    "algo_tag": ALGO_TAG,
    "run_id": RUN_ID,
    "n_clusters": int(N_CLUSTERS),
    "n_samples": int(X_scaled.shape[0]),
    "n_features": int(X_scaled.shape[1]),
    "inertia": float(kmeans.inertia_),
    "silhouette": None if np.isnan(sil) else float(sil),
    "silhouette_per_cluster": {int(k): float(v) for k, v in sil_per_cluster.items()},   # ← new
    "cluster_counts": np.bincount(labels).tolist(),
    "input_root": str(INPUT_DIR),
    "patients": PATIENT_IDS,
    "task": TASK,
    "random_state": RANDOM_STATE,
    "note": "Clustering on raw flattened ERSP (300x128) without PCA.",
}

print("Saving metrics to:", metrics_path)
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)


In [ ]:
## 7. Cluster-average ERSP heatmaps (optional but useful)

# We reuse X_3d: shape (n_samples, 300, 128)
n_clusters = N_CLUSTERS

fig, axes = plt.subplots(
    nrows=int(np.ceil(n_clusters / 4)),  # e.g., 4 per row
    ncols=4,
    figsize=(16, 3 * int(np.ceil(n_clusters / 4)))
)

axes = axes.ravel()

for c in range(n_clusters):
    ax = axes[c]
    idx_c = np.where(labels == c)[0]

    if len(idx_c) == 0:
        # Empty cluster (shouldn't happen often, but safe)
        ax.set_title(f"Cluster {c} (empty)")
        ax.axis('off')
        continue

    # Mean ERSP across samples in this cluster
    mean_ersp = X_3d[idx_c].mean(axis=0)  # shape (300, 128)

    im = ax.imshow(
        mean_ersp.T,           # freq on y-axis, time on x-axis (optional transpose)
        aspect="auto",
        origin="lower",
        interpolation="nearest"
    )
    ax.set_title(f"Cluster {c} (n={len(idx_c)})", fontsize=8)
    ax.set_xlabel("Time (samples)")
    ax.set_ylabel("Frequency bin")

    # Optional: add colorbar per subplot (can be heavy)
    # plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

# Hide unused axes if n_clusters not multiple of 4
for j in range(n_clusters, len(axes)):
    axes[j].axis('off')

plt.tight_layout()

ersp_fig_path = FIGURES_ERSP_DIR / f"{ALGO_TAG}_cluster_ERSPs_{RUN_ID}.png"
print("Saving ERSP cluster figure to:", ersp_fig_path)
plt.savefig(ersp_fig_path, dpi=150)
plt.close()


In [ ]:
## 8. Append run info to global pipeline log

log_path = LOG_DIR / "pipeline_log.txt"
log_line = (
    f"[{datetime.datetime.now().isoformat(sep=' ', timespec='seconds')}] "
    f"{SCRIPT_NAME} ({ALGO_TAG}, {RUN_ID}) -> "
    f"clusters={N_CLUSTERS}, n_samples={X_scaled.shape[0]}, "
    f"labels_file={labels_path.name}, metrics_file={metrics_path.name}\n"
)

with open(log_path, "a", encoding="utf-8") as f:
    f.write(log_line)

print("Appended to log:", log_path)
print(log_line)


# 2. GMM

In [ ]:
# ## GMM 1. Prepare feature matrix (reuse X_raw if available)

# from sklearn.preprocessing import StandardScaler
# import numpy as np
# from sklearn.mixture import GaussianMixture
# from sklearn.metrics import silhouette_score

# # We assume N_FREQ, N_TIME are already defined (e.g. 129, 300)
# N_FEATURES = N_FREQ * N_TIME
# N_CLUSTERS_LOOP = [5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24]   # same as K-means
# RANDOM_STATE = 42

# for N_CLUSTERS in N_CLUSTERS_LOOP:
#     # If X_raw does not exist yet or is out of date, rebuild it from ersp_list
#     if "X_raw" not in globals() or X_raw.shape[0] != len(ersp_list):
#         print("Building X_raw from ersp_list for GMM...")
#         n_samples = len(ersp_list)
#         X_raw = np.empty((n_samples, N_FEATURES), dtype=np.float64)
#         for i, arr in enumerate(ersp_list):
#             X_raw[i, :] = arr.reshape(-1)
#     else:
#         print("Using existing X_raw of shape:", X_raw.shape)

#     # Standardize features for GMM
#     scaler_gmm = StandardScaler()
#     X_scaled_gmm = scaler_gmm.fit_transform(X_raw.astype(np.float64))
#     print("X_scaled_gmm shape:", X_scaled_gmm.shape)

    
    
#     ## GMM 2. Fit 13-component Gaussian Mixture and compute metrics



#     gmm = GaussianMixture(
#         n_components=N_CLUSTERS,
#         covariance_type="diag",  # more tractable in high-D than full
#         random_state=RANDOM_STATE,
#         n_init=5,
#         max_iter=500,
#         verbose=1,
#     )

#     gmm.fit(X_scaled_gmm)

#     gmm_labels = gmm.predict(X_scaled_gmm)        # hard cluster assignments
#     gmm_probs  = gmm.predict_proba(X_scaled_gmm)  # soft assignments

#     cluster_counts_gmm = np.bincount(gmm_labels)
#     bic = gmm.bic(X_scaled_gmm)
#     aic = gmm.aic(X_scaled_gmm)
#     sil_gmm = (
#         silhouette_score(X_scaled_gmm, gmm_labels)
#         if len(np.unique(gmm_labels)) > 1
#         else np.nan
#     )

#     print("\n=== GMM Summary ===")
#     print("n_components (K):", N_CLUSTERS)
#     print("Cluster counts  :", cluster_counts_gmm)
#     print("BIC             :", bic)
#     print("AIC             :", aic)
#     print("Silhouette      :", sil_gmm)

#     # Attach to metadata
#     df_meta["cluster_gmm_raw"] = gmm_labels
#     df_meta["cluster_gmm_confidence"] = gmm_probs.max(axis=1)
    


#         ## GMM 3. Visualize cluster-average ERSPs (freq × time)

#     n_clusters = N_CLUSTERS

#     # Compute mean ERSP for each cluster
#     cluster_means = []
#     for c in range(n_clusters):
#         idx_c = np.where(gmm_labels == c)[0]
#         if len(idx_c) == 0:
#             cluster_means.append(None)
#             continue
#         # Stack corresponding ERSPs and average
#         arr_c = np.stack([ersp_list[i] for i in idx_c], axis=0)  # (n_c, N_FREQ, N_TIME)
#         mean_c = arr_c.mean(axis=0)
#         cluster_means.append(mean_c)

#     # Determine common color scale based on all cluster means
#     vals = []
#     for mc in cluster_means:
#         if mc is not None:
#             vals.append(mc.ravel())
#     if vals:
#         vals = np.concatenate(vals)
#         vmax = np.nanmax(np.abs(vals))
#         vmin = -vmax
#     else:
#         vmin = vmax = None

#     # Plot grid of cluster-average ERSPs
#     n_cols = 5
#     n_rows = int(np.ceil(n_clusters / n_cols))

#     fig, axes = plt.subplots(n_rows, n_cols, figsize=(3 * n_cols, 2.5 * n_rows))
#     axes = np.array(axes).reshape(-1)

#     for c in range(n_clusters):
#         ax = axes[c]
#         mean_c = cluster_means[c]
#         if mean_c is None:
#             ax.set_title(f"Cluster {c} (empty)", fontsize=8)
#             ax.axis("off")
#             continue

#         im = ax.imshow(
#             mean_c,
#             aspect="auto",
#             origin="lower",
#             cmap="bwr",
#             vmin=vmin,
#             vmax=vmax,
#             interpolation="nearest",
#         )

#         n_c = np.sum(gmm_labels == c)
#         ax.set_title(f"Cluster {c} (n={n_c})", fontsize=8)
#         ax.set_xlabel("Time (samples)", fontsize=7)
#         ax.set_ylabel("Freq", fontsize=7)

#     # Hide unused axes if any
#     for ax in axes[n_clusters:]:
#         ax.axis("off")

#     plt.tight_layout()
#     plt.show()
#     plt.close()

In [ ]:
# ## GMM 3. Visualize cluster-average ERSPs (freq × time)

# import matplotlib.pyplot as plt

# n_clusters = N_CLUSTERS

# # Compute mean ERSP for each cluster
# cluster_means = []
# for c in range(n_clusters):
#     idx_c = np.where(gmm_labels == c)[0]
#     if len(idx_c) == 0:
#         cluster_means.append(None)
#         continue
#     # Stack corresponding ERSPs and average
#     arr_c = np.stack([ersp_list[i] for i in idx_c], axis=0)  # (n_c, N_FREQ, N_TIME)
#     mean_c = arr_c.mean(axis=0)
#     cluster_means.append(mean_c)

# # Determine common color scale based on all cluster means
# vals = []
# for mc in cluster_means:
#     if mc is not None:
#         vals.append(mc.ravel())
# if vals:
#     vals = np.concatenate(vals)
#     vmax = np.nanmax(np.abs(vals))
#     vmin = -vmax
# else:
#     vmin = vmax = None

# # Plot grid of cluster-average ERSPs
# n_cols = 5
# n_rows = int(np.ceil(n_clusters / n_cols))

# fig, axes = plt.subplots(n_rows, n_cols, figsize=(3 * n_cols, 2.5 * n_rows))
# axes = np.array(axes).reshape(-1)

# for c in range(n_clusters):
#     ax = axes[c]
#     mean_c = cluster_means[c]
#     if mean_c is None:
#         ax.set_title(f"Cluster {c} (empty)", fontsize=8)
#         ax.axis("off")
#         continue

#     im = ax.imshow(
#         mean_c,
#         aspect="auto",
#         origin="lower",
#         cmap="bwr",
#         vmin=vmin,
#         vmax=vmax,
#         interpolation="nearest",
#     )

#     n_c = np.sum(gmm_labels == c)
#     ax.set_title(f"Cluster {c} (n={n_c})", fontsize=8)
#     ax.set_xlabel("Time (samples)", fontsize=7)
#     ax.set_ylabel("Freq", fontsize=7)

# # Hide unused axes if any
# for ax in axes[n_clusters:]:
#     ax.axis("off")

# cbar = fig.colorbar(im, ax=axes.tolist(), shrink=0.7, pad=0.02)
# cbar.set_label("Mean ERSP (cluster avg)", fontsize=8)

# plt.tight_layout()
# plt.show()


In [ ]:
# So you tried but you failed at The GMM. Working with ERSP clustering project on chat 